# 这里是 -> **[本篇笔记博客](https://blog.algieba12.cn/llm07-1-llama-index-workflow-groq/)**

In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# 1. 确保安装最新版 uv
!pip install -U uv

# 2. 使用修正后的包名进行安装
# llama-index 包含核心组件
# llama-index-llms-groq 是 Groq 的适配器
# llama-index-llms-gemini 是 Gemini 的适配器
!uv pip install     llama-index     llama-index-llms-groq     llama-index-llms-gemini llama-index-llms-openai-like    feedparser     requests     transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 65.4 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 114 packages in 2.01s                                       
Prepared 30 packages in 997ms                                            
Uninstalled 6 packages in 875ms
Installed 31 packages in 102ms                              
 + banks==2.4.1
 - deprecated==1.3.1
 + deprecated==1.2.18
 + dirtyjson==1.0.8
 + feedparser==6.0.12
 + griffe==2.0.0
 + griffecli==2.0.0
 + griffelib==2.0.0
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 + llama-cloud==0.1.35
 + llama-cloud-services==0.6.54
 + llama-index==0.14.15
 + llama-index-cli==0.5.4
 + llama-index-core==0.14.15
 + llama-index-embeddings-openai==0.5.2
 + llama-index-indices-managed-llama-cloud==0.9.4
 + llama-index-instrumentation==0.4.2
 + llama-index-llms-gemini==0.6.2
 + llama-index-llms-groq==0.4.1
 + llama-index-llms-openai==0.6.25
 + llama-index-llms-openai-like==0.5.3
 + llama-index-rea

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [4]:
import feedparser

def get_latest_blog_post(rss_url:str)->str:
    """
    用于请求并解析目标博客的 RSS feed，获取最新的一篇博客文章信息。
    当咱们需要检查博客是否有更新时，必须首先调用此工具。
    """
    print(f"【calling】 get_latest_blog_post with rss_url:{rss_url}")
    try:
        feed = feedparser.parse(rss_url)
        if feed.entries:
            latest_entry = feed.entries[0]
            print(f"debug: title {latest_entry.title} link:{latest_entry.link}")

            return (
                f"Title: {latest_entry.title}\n"
                f"Link: {latest_entry.link}\n\n"
            )

        return f"No posts found in rss feed {rss_url}"
    except Exception as e:
        return f"Exception: {str(e)} on fetching blog"


In [5]:
get_latest_blog_post("https://blog.algieba12.cn/atom.xml")

【calling】 get_latest_blog_post with rss_url:https://blog.algieba12.cn/atom.xml
debug: title 基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋 link:https://blog.algieba12.cn/llm07-rss-notification-react-agent/


'Title: 基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋\nLink: https://blog.algieba12.cn/llm07-rss-notification-react-agent/\n\n'

In [6]:
import requests
def send_email_notification(post_title:str, post_link:str)->str:
    """
    当发现博客有更新时，必须调用此工具发送邮件通知。
    参数 post_title: 新博客的标题
    参数 post_link: 新博客的链接
    """
    print(f"【calling】 send_email_notification with post_title {post_title} link {post_link}")
    target_email= user_secrets.get_secret("TARGET_EMAIL")
    email_api_key = user_secrets.get_secret("EMAIL_API_KEY")
    headers = {
        "Authorization": f"Bearer {email_api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "from" : "onboarding@resend.dev",
        "to":target_email,
        "subject":f"阿尔的代码屋更新咯：{post_title}",
        "text":f"检测到 阿尔的代码屋 更新了一篇新博客 \n\n 标题：{post_title}]\n 链接: {post_link}"
    }

    try:
        response = requests.post("https://api.resend.com/emails", headers=headers, json=payload)
        if response.status_code == 200:
            return "Email sent successfully via API"
        return f"Email sent failed. {response.text}"
    except Exception as e:
        return f"Exception: {str(e)} on sending email"


In [7]:
def send_all_notifications(post_title: str, post_link: str) -> str:
    """
    当发现博客有更新时，必须调用此工具。调用此工具会自动同时发送邮件和微信通知。
    参数 post_title: 新博客的标题
    参数 post_link: 新博客的链接
    """
    print(f"【Agent】 触发了联合通知工具: {post_title}")

    email_res = send_email_notification(post_title, post_link)
    # wechat_res = send_wechat_notification(post_title, post_link)

    return f"邮件通知结果: {email_res} | 微信通知结果: {wechat_res}"


In [8]:
from llama_index.core.tools import FunctionTool
tool_get_blog = FunctionTool.from_defaults(fn=get_latest_blog_post)
tool_send_all = FunctionTool.from_defaults(fn=send_all_notifications)


In [9]:
import os
from llama_index.llms.groq import Groq
from llama_index.core import Settings

os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

# 推荐使用 llama-3.3-70b，它的逻辑推理能力足以应对多智能体协作
# Settings.llm = Groq(model="llama-3.3-70b-versatile")

In [10]:
from llama_index.llms.openai_like import OpenAILike

Settings.llm = OpenAILike(
    model="Qwen/Qwen3-8B",
    api_key=user_secrets.get_secret("SILICON-API_KEY"),
    api_base="https://api.siliconflow.cn/v1",
    is_chat_model=True
)

In [11]:
from llama_index.core.workflow import Workflow, step, StartEvent, StopEvent, Context, Event

class NewBlogPostEvent(Event):
    title: str
    link: str

In [13]:
from llama_index.core.agent.workflow import ReActAgent

CHECKER_SYSTEM_PROMPT = """
### ROLE
你是一个极其精准的数据提取专家。

### TASK
1. 调用工具获取最新博客文章。
2. 提取其标题和链接。
3. **必须**以 JSON 格式输出。

### OUTPUT_FORMAT
{"title": "文章标题", "link": "文章链接"}
"""
checker = ReActAgent(
            name="checker",
            system_prompt=CHECKER_SYSTEM_PROMPT,
            tools=[tool_get_blog],
            llm=Settings.llm
        )

In [14]:
NOTIFIER_SYSTEM_PROMPT = """
### ROLE
你是一个通知官。

### TASK
请根据提供的信息发送通知。
直接调用工具发送，不要解释你的行为。发送成功后回复“已发出”即可。
"""
notifier = ReActAgent(
            name="notifier",
            system_prompt=NOTIFIER_SYSTEM_PROMPT,
            tools=[tool_send_all],
            llm=Settings.llm
        )

In [15]:
import os
import re
import json

global_last_title = ""



class DemoBlogMonitor(Workflow):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        self.checker = checker
        
        self.notifier = notifier

    @step
    async def check_step(self, ev: StartEvent) -> NewBlogPostEvent | StopEvent:
        global global_last_title
        rss_url = ev.get("rss_url")
        
        print(f"--- [Checker] 目标: {rss_url} ---")
        print(f"--- [Checker] 基准: '{global_last_title}' ---")

        response = await self.checker.run(user_msg=f"请提取此 RSS 的最新数据：{rss_url}")
        res_text = str(response)
        print(f"DEBUG [Agent Output]: {res_text}")

        try:
            json_match = re.search(r"(\{.*\})", res_text, re.DOTALL)
            if not json_match:
                raise ValueError("未在回复中找到 JSON 块")
            
            data = json.loads(json_match.group(1))
            current_title = data.get("title", "").strip()
            current_link = data.get("link", "").strip()

            if not current_title or not current_link:
                raise ValueError("JSON 数据缺失字段")

            # 在 Python 侧进行确定性比对
            if current_title != global_last_title:
                print(f"发现新动态: {current_title}")
                # 注意：这里我们只在成功触发通知时更新状态
                return NewBlogPostEvent(title=current_title, link=current_link)
            else:
                return StopEvent(result=f"判定：内容未更新")

        except Exception as e:
            # 如果 JSON 解析失败，尝试最后的正则兜底
            print(f"WARN: JSON 解析失败 ({e}), 尝试正则兜底...")
            link_match = re.search(r"(https?://\S+)", res_text)
            if link_match:
                link = link_match.group(1).strip(" ”\"。")
                # 假设引号中间的内容是标题
                title_match = re.search(r"[是“\"'](.*?)[”\"']", res_text)
                title = title_match.group(1) if title_match else "未识别标题"
                
                if title != global_last_title:
                    return NewBlogPostEvent(title=title, link=link)
            
            return StopEvent(result=f"解析彻底失败。原文: {res_text}")

    @step
    async def notify_step(self, ev: NewBlogPostEvent) -> StopEvent:
        global global_last_title
        print(f"--- [Notifier] 正在发送: {ev.title} ---")
        
        await self.notifier.run(user_msg=f"发送通知：标题《{ev.title}》，链接 {ev.link}")
        
        # 只有通知成功发出后，才真正更新全局基准
        global_last_title = ev.title
        return StopEvent(result=f"完成。新基准已设为: {ev.title}")





In [ ]:
async def main():
    monitor = DemoBlogMonitor(timeout=120)
    
    print("\n>>> Run #1")
    await monitor.run(rss_url="https://blog.algieba12.cn/atom.xml")
    
    print("\n>>> Run #2")
    await monitor.run(rss_url="https://blog.algieba12.cn/atom.xml")

In [16]:
await main()


>>> Run #1
--- [Checker] 目标: https://blog.algieba12.cn/atom.xml ---
--- [Checker] 基准: '' ---
【calling】 get_latest_blog_post with rss_url:https://blog.algieba12.cn/atom.xml
debug: title 基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋 link:https://blog.algieba12.cn/llm07-rss-notification-react-agent/
DEBUG [Agent Output]: 最新的博客文章标题是“基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋”，链接为 https://blog.algieba12.cn/llm07-rss-notification-react-agent/。
WARN: JSON 解析失败 (未在回复中找到 JSON 块), 尝试正则兜底...
--- [Notifier] 正在发送: “基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋 ---

>>> Run #2
--- [Checker] 目标: https://blog.algieba12.cn/atom.xml ---
--- [Checker] 基准: '“基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋' ---
【calling】 get_latest_blog_post with rss_url:https://blog.algieba12.cn/atom.xml
debug: title 基于 LlamaIndex ReAct 框架手搓全自动博客监控 Agent - 大模型实战 07 | 阿尔的代码屋 link:https://blog.algieba12.cn/llm07-rss-notification-react-agent/
DEBUG [Agent Output]: 最新的博客文章标题是“